# 注意力机制：MHA、MQA、GQA 与 MLA

> **难度：** 中级 | **时间：** 约50分钟

注意力机制的选择是 LLM 推理服务中 KV-cache 内存优化最关键的一环。本 notebook 从多头注意力（MHA）出发，逐步演进到多潜变量注意力（MLA），展示每种变体如何在内存、质量和计算复杂度之间做取舍。

**你将学到：**
- MHA 如何为每个头分别计算 K/V
- MQA 如何让所有头共享单一 K/V（Falcon）
- GQA 如何通过分组平衡内存与质量（LLaMA-2/3）
- MLA 如何通过低秩投影压缩 K/V（DeepSeek-V2）
- 基于真实模型配置的 KV-cache 定量对比
- 其他变种：线性注意力、滑动窗口注意力、稀疏注意力
- 各变体在张量并行（TP）下的不同策略（切分 vs 复制 KV 头）

**前置知识：** [00-kv-cache](00-kv-cache.ipynb)（理解 KV-cache 的概念及其重要性），[02-tensor-parallelism](../../02-tensor-parallelism.ipynb)（第 7 节需要）

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import sys, os

sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))

from mp_tutorial.inference import (
    calc_kv_cache_size,
    attention_forward_sim,
)
from mp_tutorial.inference_viz import (
    draw_attention_head_layout,
    draw_kv_cache_comparison_bar,
    draw_mla_projection_flow,
    draw_mha_vs_mqa_vs_gqa,
)
from mp_tutorial.formatting import info_box, comparison_table

import warnings
warnings.filterwarnings('ignore')

from mp_tutorial.fonts import configure_cjk_fonts
configure_cjk_fonts()

torch.manual_seed(42)
print('Ready!')

## 1. 多头注意力（MHA）—— 基准线

标准的多头注意力（Vaswani et al., 2017）将输入分别投影为**每个头独立的 Q、K、V**。各头独立计算注意力后拼接输出。

$$\text{head}_i = \text{Softmax}\left(\frac{Q_i K_i^T}{\sqrt{d_k}}\right) V_i$$

$$\text{MHA}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

在自回归解码时，所有过去 token 的 K 和 V 都会被缓存。KV-cache 每个 token 每层存储 **2 个张量（K 和 V） × n_heads × head_dim** 个值。

In [ ]:
# 小型 MHA 示例：4 个头，d_model=16，seq_len=4
batch, seq_len, d_model = 1, 4, 16
n_heads = 4
head_dim = d_model // n_heads  # 4

x = torch.randn(batch, seq_len, d_model)

result = attention_forward_sim("mha", x, n_heads=n_heads, n_kv_heads=n_heads, head_dim=head_dim)

print("=== MHA 张量形状 ===")
print(f"输入:     {list(x.shape)}           (batch, seq_len, d_model)")
print(f"Q:        {list(result['q'].shape)}        (batch, n_heads, seq_len, head_dim)")
print(f"K:        {list(result['k'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"V:        {list(result['v'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"分数:     {list(result['scores'].shape)}      (batch, n_heads, seq_len, seq_len)")
print(f"输出:     {list(result['output'].shape)}       (batch, seq_len, n_heads * head_dim)")
print()
print(f"每 token 的 KV cache: 2 × {n_heads} 个头 × {head_dim} 维 = {2 * n_heads * head_dim} 个值")

In [ ]:
# 可视化 MHA 头部布局
draw_attention_head_layout("mha", n_q_heads=4, n_kv_heads=4, head_dim=4)
plt.show()

In [ ]:
# 大规模下的 KV cache 开销：LLaMA-2-70B 配置（假设使用 MHA）
mha_cache = calc_kv_cache_size(
    variant="mha", n_heads=64, n_kv_heads=64,
    head_dim=128, seq_len=4096, n_layers=80
)
print(f"LLaMA-2-70B（若使用 MHA）：4096 token 的 KV-cache = {mha_cache / 1024**3:.2f} GB")
print(f"这只是单个请求！32 个请求的批次仅 KV-cache 就需要 {32 * mha_cache / 1024**3:.1f} GB。")

**核心洞察：** MHA 的 KV-cache 按 `2 × n_layers × n_heads × head_dim × seq_len` 增长。对于头数众多的大模型，这往往成为推理服务中的主要内存瓶颈——在长序列场景下甚至超过模型权重本身。

接下来的三种变体都致力于**减少 KV-cache 中存储的内容**。

## 2. 多查询注意力（MQA）—— 所有头共享 K/V

多查询注意力（Shazeer, 2019）采用最简单的缩减方式：**所有查询头共享同一个 K 和 V**。

$$Q_i = X W_i^Q \quad (\text{每个头独立})$$
$$K = X W^K, \quad V = X W^V \quad (\text{所有头共享})$$

这将 KV-cache 缩减了 `n_heads` 倍——从 64 对 K/V 减少到仅 1 对。K/V 投影矩阵也更小，略微减少了参数量。

**使用该方案的模型：** Falcon-40B、PaLM、StarCoder

In [ ]:
# 小型 MQA 示例：4 个查询头，1 个 KV 头
result_mqa = attention_forward_sim("mqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)

print("=== MQA 张量形状 ===")
print(f"Q:        {list(result_mqa['q'].shape)}        (batch, n_heads, seq_len, head_dim)  — 每个头仍然独立")
print(f"K:        {list(result_mqa['k'].shape)}        (batch, 1, seq_len, head_dim)  — 单个共享 K")
print(f"V:        {list(result_mqa['v'].shape)}        (batch, 1, seq_len, head_dim)  — 单个共享 V")
print(f"输出:     {list(result_mqa['output'].shape)}       (batch, seq_len, d_model)")
print()
print(f"每 token 的 KV cache: 2 × 1 个头 × {head_dim} 维 = {2 * 1 * head_dim} 个值")
print(f"相比 MHA 的缩减倍数: {n_heads}×！")

In [ ]:
# 可视化：MQA 中广播机制的工作方式
draw_attention_head_layout("mqa", n_q_heads=4, n_kv_heads=1, head_dim=4)
plt.show()

In [ ]:
# 大规模 MQA cache 节省：Falcon-40B 配置
mqa_cache = calc_kv_cache_size(
    variant="mqa", n_heads=64, n_kv_heads=1,
    head_dim=128, seq_len=4096, n_layers=60
)
print(f"Falcon-40B (MQA): 4096 token 的 KV-cache = {mqa_cache / 1024**3:.4f} GB")
print(f"相同配置下 MHA: {calc_kv_cache_size('mha', 64, 64, 128, 4096, 60) / 1024**3:.2f} GB")
print(f"缩减倍数: {64}×")

**权衡：** MQA 提供了巨大的内存节省，但可能影响模型质量。只有一个 K/V 表示意味着所有头被迫在相同的键值空间上进行注意力计算——它们失去了专门化的能力。实践中，从头训练的 MQA 模型效果不错（Falcon、PaLM），但将现有 MHA 模型微调为 MQA 可能导致质量下降。

这激发了 GQA 的诞生：MHA 和 MQA 之间的折中方案。

## 3. 分组查询注意力（GQA）—— 折中方案

分组查询注意力（Ainslie et al., 2023）将查询头分成**若干组**，每组共享一个 K/V 头。设有 `n_kv_heads` 个组：

- `n_kv_heads = n_heads` → 等价于 **MHA**（每个头都有独立的 K/V）
- `n_kv_heads = 1` → 等价于 **MQA**（所有头共享一个 K/V）
- `1 < n_kv_heads < n_heads` → **GQA**（分组共享）

$$\text{group}(i) = \lfloor i \cdot n_{kv} / n_q \rfloor$$

**使用该方案的模型：** LLaMA-2-70B（8 个 KV 头）、LLaMA-3（8 个 KV 头）、Mistral（8 个 KV 头）

In [ ]:
# 小型 GQA 示例：4 个查询头，2 个 KV 头（每组 2 个）
result_gqa = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=2, head_dim=head_dim)

print("=== GQA 张量形状（4 个 Q 头，2 个 KV 头）===")
print(f"Q:        {list(result_gqa['q'].shape)}        (batch, n_q_heads, seq_len, head_dim)")
print(f"K:        {list(result_gqa['k'].shape)}        (batch, n_kv_heads, seq_len, head_dim)  — 2 个 KV 头")
print(f"V:        {list(result_gqa['v'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"输出:     {list(result_gqa['output'].shape)}       (batch, seq_len, d_model)")
print()
print(f"Q 头 0,1 共享 KV 头 0 | Q 头 2,3 共享 KV 头 1")
print(f"每 token 的 KV cache: 2 × 2 个头 × {head_dim} 维 = {2 * 2 * head_dim} 个值")
print(f"相比 MHA 的缩减倍数: {n_heads // 2}×")

In [ ]:
# 可视化 GQA 分组
draw_attention_head_layout("gqa", n_q_heads=8, n_kv_heads=2, head_dim=128)
plt.show()

In [ ]:
# 验证：GQA 可以退化为 MHA 和 MQA
print("GQA 作为一般化框架：")
print()

# n_kv_heads = n_heads 时 -> MHA
result_gqa_as_mha = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=4, head_dim=head_dim)
result_pure_mha = attention_forward_sim("mha", x, n_heads=4, n_kv_heads=4, head_dim=head_dim)
match_mha = torch.allclose(result_gqa_as_mha['output'], result_pure_mha['output'], atol=1e-6)
print(f"GQA(n_kv=n_q=4) == MHA: {match_mha}")

# n_kv_heads = 1 时 -> MQA
result_gqa_as_mqa = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)
result_pure_mqa = attention_forward_sim("mqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)
match_mqa = torch.allclose(result_gqa_as_mqa['output'], result_pure_mqa['output'], atol=1e-6)
print(f"GQA(n_kv=1)    == MQA: {match_mqa}")

In [ ]:
# 使用现有可视化函数概览三种方案
draw_mha_vs_mqa_vs_gqa(n_q_heads=8, head_dim=128)
plt.show()

**核心要点：** GQA 提供了一个在内存和质量之间可调节的旋钮。LLaMA-2-70B 使用 64 个查询头但只有 8 个 KV 头——KV-cache 缩减 8 倍的同时保留了 MHA 的大部分质量。这已成为大模型的行业标准配置。

但我们能做得更好吗？如果不是减少 KV 头的*数量*，而是压缩*表示本身*呢？

## 4. 多潜变量注意力（MLA）—— 低秩 KV 压缩

多潜变量注意力（DeepSeek-AI, 2024）采用了根本不同的方法。MLA 不是让查询组共享 KV 头，而是在缓存之前将 KV 表示**压缩**到低秩潜空间。

核心思路：
1. **下投影**：将输入压缩为潜变量：$c = X W_{\text{down}}$，其中 $c \in \mathbb{R}^{d_{\text{compressed}}}$
2. **缓存**：只缓存压缩后的潜变量 $c$（远小于独立的 K、V）
3. **上投影**：计算注意力时还原为 K 和 V：$K = c W_{\text{up}}^K$，$V = c W_{\text{up}}^V$

缓存中每个 token 只存储一个 $d_{\text{compressed}}$ 维向量，而非分别存储 $n_{\text{heads}} \times d_{\text{head}}$ 维的 K 和 V。

**使用该方案的模型：** DeepSeek-V2、DeepSeek-V3

In [ ]:
# 小型 MLA 示例：将 d_model=16 压缩到 d_compressed=6
d_compressed = 6

result_mla = attention_forward_sim(
    "mla", x, n_heads=4, n_kv_heads=4, head_dim=head_dim, d_compressed=d_compressed
)

print("=== MLA 张量形状 ===")
print(f"Q:              {list(result_mla['q'].shape)}        (batch, n_heads, seq_len, head_dim)")
print(f"压缩后的 KV:    {list(result_mla['compressed_kv'].shape)}       (batch, seq_len, d_compressed) ← 这个被缓存")
print(f"K（解压后）:    {list(result_mla['k'].shape)}        (batch, n_heads, seq_len, head_dim) — 重建的")
print(f"V（解压后）:    {list(result_mla['v'].shape)}        (batch, n_heads, seq_len, head_dim) — 重建的")
print(f"输出:           {list(result_mla['output'].shape)}       (batch, seq_len, d_model)")
print()
mha_cache_per_token = 2 * n_heads * head_dim
mla_cache_per_token = d_compressed
print(f"MHA 缓存: 2 × {n_heads} × {head_dim} = {mha_cache_per_token} 个值/token")
print(f"MLA 缓存: {d_compressed} 个值/token（压缩潜变量）")
print(f"压缩比: {mha_cache_per_token / mla_cache_per_token:.1f}×")

In [ ]:
# 可视化 MLA 流水线
draw_mla_projection_flow(d_model=512, d_compressed=64, n_heads=8, head_dim=64)
plt.show()

In [ ]:
# 大规模 MLA cache 节省：DeepSeek-V2 配置
# DeepSeek-V2: 128 个头, head_dim=128, 60 层, d_compressed=512
mla_cache = calc_kv_cache_size(
    variant="mla", n_heads=128, n_kv_heads=128,
    head_dim=128, seq_len=4096, n_layers=60, d_compressed=512
)
mha_equivalent = calc_kv_cache_size(
    variant="mha", n_heads=128, n_kv_heads=128,
    head_dim=128, seq_len=4096, n_layers=60
)
print(f"DeepSeek-V2 (MLA, d_c=512): KV-cache = {mla_cache / 1024**3:.4f} GB")
print(f"相同模型使用 MHA:           KV-cache = {mha_equivalent / 1024**3:.2f} GB")
print(f"缩减倍数: {mha_equivalent / mla_cache:.0f}×")

**权衡：** MLA 实现了极端的压缩，但增加了计算开销——从压缩潜变量到 K/V 的上投影在每次注意力计算时都要执行。这是一种用计算换内存的策略：额外的矩阵乘法换来大幅缩小的 KV-cache。对于内存受限的推理工作负载（长序列、大批次），这种交易是非常划算的。

## 5. 综合对比

让我们使用真实模型配置，将四种变体进行并排对比。

In [ ]:
# 使用真实模型配置进行并排 KV cache 对比
configs = [
    {"name": "MHA\n(假设 70B)", "variant": "mha",
     "n_heads": 64, "n_kv_heads": 64, "head_dim": 128, "n_layers": 80},
    {"name": "MQA\n(Falcon-40B)", "variant": "mqa",
     "n_heads": 64, "n_kv_heads": 1, "head_dim": 128, "n_layers": 60},
    {"name": "GQA\n(LLaMA-2-70B)", "variant": "gqa",
     "n_heads": 64, "n_kv_heads": 8, "head_dim": 128, "n_layers": 80},
    {"name": "MLA\n(DeepSeek-V2)", "variant": "mla",
     "n_heads": 128, "n_kv_heads": 128, "head_dim": 128, "n_layers": 60,
     "d_compressed": 512},
]

draw_kv_cache_comparison_bar(configs)
plt.show()

In [ ]:
# 总结表格
print("┌──────────┬──────────────┬───────────────┬──────────────────────────────────────┐")
print("│ 变体     │ KV 头数      │ 缓存/Token    │ 核心权衡                             │")
print("├──────────┼──────────────┼───────────────┼──────────────────────────────────────┤")
print("│ MHA      │ n_heads      │ 2·h·d         │ 完整质量，完整内存开销               │")
print("│ MQA      │ 1            │ 2·d           │ 最大节省，质量有所损失               │")
print("│ GQA      │ g（可调）    │ 2·g·d         │ 平衡方案——行业标准                   │")
print("│ MLA      │ 无（潜变量） │ d_compressed  │ 最大压缩，额外计算开销               │")
print("└──────────┴──────────────┴───────────────┴──────────────────────────────────────┘")
print()
print("h = n_heads, d = head_dim, g = n_kv_groups")
print()
print("演进路线: MHA (2017) → MQA (2019) → GQA (2023) → MLA (2024)")
print("每一步都通过某种取舍来减少 KV-cache 内存。")

## 6. 其他注意力变种：超越 Softmax

MHA→MQA→GQA→MLA 这一族都使用标准的 softmax 注意力——它们减少的是**缓存的内容**，而非注意力的计算方式。另一条研究路线直接攻击注意力 O(n²) 的计算复杂度。

### 线性注意力（Linear Attention）

标准注意力计算 $\text{Softmax}(QK^T)V$，需要实例化一个 $n \times n$ 的注意力矩阵。线性注意力（Katharopoulos et al., 2020）用核函数 $\phi$ 替换 softmax：

$$\text{标准：} \quad O = \text{Softmax}(QK^T) \cdot V \qquad \mathcal{O}(n^2 d)$$
$$\text{线性：} \quad O = \phi(Q) \cdot (\phi(K)^T V) \qquad \mathcal{O}(n d^2)$$

技巧在于**结合律**：先计算 $\phi(K)^T V$（一个 $d \times d$ 矩阵），就完全避免了 $n \times n$ 的乘积。这使注意力在序列长度上为 **O(n)**，但常数因子 $d^2$ 意味着只在非常长的序列上才有优势。

**后续发展：** RetNet（多尺度保留机制）、RWKV（线性递归作为注意力）、Mamba/S4（状态空间模型——严格来说不是"注意力"，但解决同样的问题）。这些架构能以 O(n) 时间处理序列，并维护**固定大小的状态**而非不断增长的 KV-cache。

### 滑动窗口注意力（Sliding Window Attention）

滑动窗口注意力（Beltagy et al., 2020；用于 Mistral、Gemma-2）限制每个 token 只在固定大小 $w$ 的窗口内进行注意力计算：

$$\text{Attn}(q_i) = \text{Softmax}(q_i \cdot K[i-w:i]^T) \cdot V[i-w:i]$$

这为 KV-cache 设置了**硬上界**：无论总序列多长，缓存最多 $w$ 个 token。窗口外的信息通过层叠间接传播：$L$ 层 × 窗口 $w$ = 有效感受野 $L \times w$ 个 token。

**使用该方案的模型：** Mistral-7B（$w = 4096$）、Gemma-2（全局层与滑动窗口层交替）

### 稀疏注意力（Sparse Attention）

Longformer 和 BigBird 结合**局部窗口**（每个 token 看邻居）与**全局 token**（少数 token 看全部）。这在保留长距离依赖的同时实现 O(n) 复杂度。

**关键洞察：** 这些方法与 MQA/GQA/MLA **基本正交**。你可以将滑动窗口与 GQA 组合（Mistral 正是这样做的），也可以与 MLA 组合。它们减少的是不同的开销：头共享减少每 token 的 KV-cache *大小*，而稀疏/线性方法减少参与注意力计算的 *token 数量*。

## 7. 注意力变体与张量并行

大模型多卡推理时，张量并行（TP）将每一层切分到多个 GPU 上。对于注意力层，Megatron-LM 的标准做法是 **Q/K/V 列并行投影**（按头切分），加上**输出投影行并行** 与一次 AllReduce。但不同注意力变体与 TP 的交互方式差异很大。

### MHA + TP：最干净的情况

`n_heads` 个独立的 Q、K、V 头，TP 切分直截了当：
- 切分：每个 GPU 分到 `n_heads / tp_size` 个完整的头（Q、K、V 各一份）
- 计算：各 GPU 独立做自己的注意力——**零通信**
- 合并：行并行 $W_O$ 后一次 AllReduce

64 头分到 8 卡 → 每卡 8 头，整除，完美。

### MQA + TP：复制问题

MQA 只有 **1 个 KV 头**，但有 `n_heads` 个 Q 头。1 个头没法切：

1. **每卡复制 KV**（标准做法）：每个 GPU 持有完整的单个 K/V 头，加上自己的那部分 Q 头。反正只有 1 个头，复制开销微不足道。
2. **从一张卡广播 KV**：rank 0 计算 KV 后广播。多一次通信，但避免冗余计算。

实践中用方案 1，因为单个 KV 头太小了，复制比广播延迟还便宜。

### GQA + TP：取决于比例

关键问题：**`n_kv_heads` 能否被 `tp_size` 整除？**

| 场景 | 例子 | 策略 |
|------|------|------|
| `n_kv_heads >= tp_size` 且整除 | 8 KV 头 on 8 GPU | 切分：每卡 1 个 KV 头 |
| `n_kv_heads >= tp_size` 但不整除 | 8 KV 头 on 6 GPU | 无法均分——避免此配置 |
| `n_kv_heads < tp_size` | 8 KV 头 on 16 GPU | **复制**：每个 KV 头复制到 `tp_size / n_kv_heads` 张卡 |

LLaMA-2-70B 的 8 个 KV 头在 8 卡上是甜点配置：每卡恰好 1 个 KV 头，8 个 Q 头在本地共享它。

**实用建议：** 设计新模型时，`n_kv_heads` 要选能整除目标 `tp_size` 的值。这就是为什么 8 KV 头如此常见——它能整除 8、16、32、64 卡。

### MLA + TP：切分上投影

MLA 的 TP 策略根本不同，因为没有 KV "头"可切：

1. **下投影** $W_{\text{down}}$：产生共享的压缩潜变量 $c$。必须完整计算（或复制），因为所有头都需要它。
2. **压缩潜变量**：在所有 GPU 上复制（很小：每 token 只有 `d_compressed`）。
3. **上投影** $W_{\text{up}}^K$、$W_{\text{up}}^V$：列并行——每个 GPU 上投影到自己负责的那部分头。
4. **注意力 + 输出投影**：从这里开始和 MHA 一样——每卡有自己的局部头。

压缩潜变量的复制正是 MLA 在 TP 下的优势：不需要复制 `2 × n_heads × head_dim` 每 token（MHA），只需复制 `d_compressed`——节省 cache 的同一压缩机制也减少了 TP 通信量。

In [ ]:
# 各变体的 TP 策略一览
print("┌──────────┬────────────────────┬──────────────────────┬─────────────────────────────┐")
print("│ 变体     │ Q 切分             │ KV 切分              │ TP 注意事项                 │")
print("├──────────┼────────────────────┼──────────────────────┼─────────────────────────────┤")
print("│ MHA      │ n_heads / tp_size  │ n_heads / tp_size    │ 无——干净切分                │")
print("│ MQA      │ n_heads / tp_size  │ 复制（1 个头）       │ KV 复制而非切分             │")
print("│ GQA      │ n_heads / tp_size  │ n_kv / tp_size *     │ * n_kv < tp 时需复制        │")
print("│ MLA      │ n_heads / tp_size  │ 复制潜变量           │ 上投影为列并行              │")
print("└──────────┴────────────────────┴──────────────────────┴─────────────────────────────┘")
print()

# 具体示例：LLaMA-2-70B GQA 在不同 TP 规模下的情况
n_q, n_kv = 64, 8
for tp in [1, 2, 4, 8, 16]:
    q_per_gpu = n_q // tp
    if n_kv >= tp:
        kv_per_gpu = n_kv // tp
        kv_strategy = f"每卡 {kv_per_gpu} 个 KV 头（切分）"
    else:
        replicas = tp // n_kv
        kv_strategy = f"每个 KV 头复制到 {replicas} 张卡（复制）"
    print(f"  TP={tp:2d}: 每卡 {q_per_gpu:2d} 个 Q 头, {kv_strategy}")

## 总结

| 变体 | 论文 | 缓存公式 | 真实案例 | 相对缓存大小 |
|------|------|----------|----------|-------------|
| **MHA** | Vaswani 2017 | `2 · n_heads · d_head` / token / 层 | — | 1×（基准） |
| **MQA** | Shazeer 2019 | `2 · d_head` / token / 层 | Falcon-40B, PaLM | 1/n_heads × |
| **GQA** | Ainslie 2023 | `2 · n_kv_heads · d_head` / token / 层 | LLaMA-2-70B (8 KV) | 1/group_size × |
| **MLA** | DeepSeek 2024 | `d_compressed` / token / 层 | DeepSeek-V2 (512) | d_c / (2·h·d) × |

**核心要点：**
- KV-cache 内存是大规模 LLM 推理服务的瓶颈
- GQA 是当前行业标准（LLaMA、Mistral、Gemma）——简单、有效、可调
- MLA 通过低秩投影进一步压缩，代价是额外的计算开销
- 选择取决于部署约束：内存受限 → 倾向 MLA/MQA，计算受限 → MHA/GQA 即可

**延伸阅读：**
- [00-kv-cache](00-kv-cache.ipynb) — KV-cache 是什么以及为什么重要
- [01-flash-attention](01-flash-attention.ipynb) — 高效注意力*计算*（与头共享正交）
- [03-paged-attention](03-paged-attention.ipynb) — 高效注意力*内存管理*
- [08-serving-architecture](08-serving-architecture.ipynb) — 这些选择如何影响推理系统设计